# Notebook 01.2 — Aquisição da infraestrutura viária e edificações (Overture Maps)

**Projeto:** Acessibilidade Geográfica às UBS de Teresina — roteiro computacional AE2SFCA  
**Programa:** MAPEPROF — Mestrado Profissional em Planejamento Urbano e Regional / IFPI  
**Autor:** Felipe Ramos Dantas  
**Orientador:** Prof. Dr. Antonio Joaquim da Silva  
**Coorientador:** Prof. Dr. Reurysson Chagas de Sousa Morais  
**Repositório:** https://github.com/felipedantas-pi/ae2sfca-ubs  
**Apêndice:** B · **Última atualização:** 2026-06-10

## Objetivo

Utilizar a área de estudo pré-processada no **Notebook 01.1** (zona urbana + *buffer* de 5 km) para extrair da **Overture Maps Foundation** os dados de infraestrutura urbana de Teresina: a malha viária (*segment* e *connector*) e as pegadas de construção (*building*). A versão (*release*) é fixada para garantir a reprodutibilidade.

## Entrada

| Arquivo | Origem |
|---|---|
| `teresina_zonaUrbana_utm.parquet` | NB 01.1 |
| `teresina_zonaUrbana_buffer5kClip_utm.parquet` | NB 01.1 |
| Overture Maps Foundation (*release* 2026-06-17.0) | API pública (download automático) |

## Saídas

Gravadas em `dados/externos/overture/`:

| Arquivo | Consumido por | Descrição |
|---|---|---|
| `zonaUrbana_5km_segmentos.geojson` / `.parquet` | NB 02.1 | Segmentos viários brutos (arestas do grafo) |
| `zonaUrbana_5km_conectores.geojson` / `.parquet` | NB 02.1 | Conectores viários brutos (nós do grafo) |
| `zonaUrbana_building_utm.geojson` | NB 05.2 | Pegadas de construção (demanda dasimétrica) |

## Pré-requisitos

- **NB 01.1** executado (recortes territoriais em `dados/externos/ibge/`).
- Conexão de internet estável (o download da Overture pode levar minutos).

---

## 1. Configuração e importações

Bibliotecas (`city2graph`, `overturemaps`), caminhos e CRS centralizados. A API da Overture exige coordenadas em WGS 84 (EPSG:4326).

In [1]:
# ── 1. IMPORTAÇÕES E CONFIGURAÇÃO GLOBAL ────────────────────────────────────
import geopandas as gpd
import city2graph as c2g
from overturemaps import geodataframe as overture_gdf

# Caminhos e CRS centralizados (ver src/mapeprof/config.py)
from mapeprof.config import (
    EXT_IBGE,        # dados/externos/ibge     — recortes do NB 01.1
    EXT_OVERTURE,    # dados/externos/overture — saídas deste notebook
    CRS_METRICO,     # EPSG:31983 — SIRGAS 2000 / UTM 23S
    CRS_WGS84,       # EPSG:4326  — exigido pela API da Overture
    criar_diretorios,
)

criar_diretorios()
print("Dependências e diretórios prontos.")
print(f"city2graph: {c2g.__version__ if hasattr(c2g, '__version__') else 'development'}")

Dependências e diretórios prontos.
city2graph: 0.3.1


## 2. Recortes da área de estudo (NB 01.1)

Carga dos polígonos da zona urbana e da área de estudo (*buffer* de 5 km) e reprojeção para WGS 84, exigido pela API da Overture.

In [2]:
# ── 2. IMPORTAÇÃO DOS DADOS DO NOTEBOOK 01.1 ────────────────────────────────
print("🗺️ Carregando recortes espaciais gerados no Notebook 01.1...")

# Polígonos em UTM (EPSG:31983), gravados pelo NB 01.1
gdf_zonaUrbana   = gpd.read_parquet(EXT_IBGE / "teresina_zonaUrbana_utm.parquet")
gdf_zonaUrbanab5kms = gpd.read_parquet(EXT_IBGE / "teresina_zonaUrbana_buffer5kClip_utm.parquet")

# A API da Overture só aceita WGS 84 — reprojeta antes da requisição
gdf_zonaUrbana_wgs   = gdf_zonaUrbana.to_crs(CRS_WGS84)
gdf_zonaUrbana_buffer5km_wgs = gdf_zonaUrbanab5kms.to_crs(CRS_WGS84)

print("✅ Recortes carregados e reprojetados para WGS 84.")

🗺️ Carregando recortes espaciais gerados no Notebook 01.1...
✅ Recortes carregados e reprojetados para WGS 84.


## 3. Malha viária: segmentos e conectores

Extração da malha viária (*segment* e *connector*) da Overture na área de estudo, com o *release* fixado, e exportação em GeoJSON (preserva os atributos aninhados) e Parquet (leitura rápida).

In [5]:
# ── 3. DOWNLOAD SEGMENTOS E CONECTORES ──────────────────────────────────────
subdatasets = ["segment", "connector"]
RELEASE_OVERTURE = '2026-06-17.0'   # fixa a versão p/ reprodutibilidade da dissertação

In [6]:
print(f"🛣️ Extraindo malha viária (release {RELEASE_OVERTURE})...")

dados_viarios = c2g.load_overture_data(
    area=gdf_zonaUrbana_buffer5km_wgs,       # área de estudo (ZU + buffer 5 km)
    types=subdatasets,
    output_dir=EXT_OVERTURE,
    prefix='zonaUrbana_5km_',
    save_to_file=False,            # mantém em memória para reprojetar antes de salvar
    return_data=True,
    release=RELEASE_OVERTURE,
    use_stac=False,
)

print("🔄 Reprojetando para UTM e salvando...")
segments_metric   = dados_viarios['segment'].to_crs(CRS_METRICO)
connectors_metric = dados_viarios['connector'].to_crs(CRS_METRICO)

# GeoJSON: formato seguro p/ as listas aninhadas da Overture (consumido pelo NB 02.1)
segments_metric.to_file(EXT_OVERTURE / "zonaUrbana_5km_segmentos.geojson", driver="GeoJSON")
connectors_metric.to_file(EXT_OVERTURE / "zonaUrbana_5km_conectores.geojson", driver="GeoJSON")

# Parquet: leitura rápida para análises locais
segments_metric.to_parquet(EXT_OVERTURE / "zonaUrbana_5km_segmentos.parquet", index=False)
connectors_metric.to_parquet(EXT_OVERTURE / "zonaUrbana_5km_conectores.parquet", index=False)

print(f"✅ {len(segments_metric):,} segmentos e {len(connectors_metric):,} conectores salvos.")

🛣️ Extraindo malha viária (release 2026-06-17.0)...
🔄 Reprojetando para UTM e salvando...
✅ 46,970 segmentos e 35,183 conectores salvos.


## 4. Pegadas de construção

Download das edificações (*building*) pela *bounding box* da zona urbana, exportadas em GeoJSON para o teste de robustez dasimétrica da demanda (NB 05.2).

In [ ]:
# ── 4. DOWNLOAD DAS PEGADAS DE CONSTRUÇÃO ────────────────────────────────────
print("🏠 Calculando os limites (bounding box) da zona urbana para a extração...")

# Retângulo envolvente da zona urbana [minX, minY, maxX, maxY], em WGS 84
bounds = gdf_zonaUrbana_wgs.total_bounds
bbox_zonaUrbana = (bounds[0], bounds[1], bounds[2], bounds[3])

print(f"📡 Solicitando 'buildings' para a bounding box: {bbox_zonaUrbana}")
gdf_building = overture_gdf("building", bbox=bbox_zonaUrbana, release=RELEASE_OVERTURE)
gdf_building.set_crs("EPSG:4326", inplace=True)
print(f"✅ Download concluído. {len(gdf_building):,} edificações mapeadas.")

# Exporta em GeoJSON (preserva os atributos aninhados da Overture)
print("💾 Reprojetando para UTM e exportando...")
caminho_buildings = EXT_OVERTURE / "zonaUrbana_building_utm.geojson"
gdf_building.to_crs(CRS_METRICO).to_file(caminho_buildings, driver="GeoJSON")
print(f"💾 Salvo em: {caminho_buildings}")

---
## Próximos passos

Os segmentos e conectores alimentam o **NB 02.1** (limpeza e correção topológica da malha viária); as pegadas de construção alimentam o **NB 05.2** (robustez dasimétrica da demanda).